# 01 — Fetch Training Data
Pull 6 months of 1-minute OHLCV bars from Alpaca Data API v2 for all 100 S&P 100 tickers.

**Prerequisites:**
1. Mount your Google Drive.
2. Add `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` to Colab Secrets (🔑 icon in left sidebar).
3. Run all cells top-to-bottom.

**Output:** One `.parquet` file per ticker at `/content/drive/MyDrive/algo_trader/data/raw/{TICKER}.parquet`

In [ ]:
# Install Alpaca SDK and parquet support
!pip install -q alpaca-py pyarrow pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RAW_DATA_DIR = '/content/drive/MyDrive/algo_trader/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
print(f'Output directory: {RAW_DATA_DIR}')

In [ ]:
# Load API credentials from Colab Secrets (never hard-code these)
from google.colab import userdata
ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError('Add ALPACA_API_KEY and ALPACA_SECRET_KEY to Colab Secrets first.')
print('Credentials loaded ✓')

In [ ]:
# S&P 100 ticker universe — must match tickers.py
SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]
print(f'Universe: {len(SP100_TICKERS)} tickers')

In [ ]:
import time
import pandas as pd
from datetime import datetime, timedelta
from tqdm.notebook import tqdm

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

END_DATE   = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0)
START_DATE = END_DATE - timedelta(days=182)   # ~6 months

FORWARD_FILL_LIMIT = 5   # Forward-fill gaps up to 5 bars
MAX_RETRIES = 3
RETRY_BACKOFF_BASE = 2   # Seconds (exponential: 2, 4, 8)

summary = []
failed  = []

for ticker in tqdm(SP100_TICKERS, desc='Fetching bars'):
    out_path = f'{RAW_DATA_DIR}/{ticker}.parquet'

    if os.path.exists(out_path):
        print(f'{ticker}: already exists — skipping.')
        continue

    df = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = StockBarsRequest(
                symbol_or_symbols=ticker,
                timeframe=TimeFrame.Minute,
                start=START_DATE,
                end=END_DATE,
                feed='iex',       # Free IEX feed; switch to 'sip' with paid plan
            )
            bars = client.get_stock_bars(request)
            df = bars.df

            # Flatten multi-index returned by alpaca-py
            if isinstance(df.index, pd.MultiIndex):
                df = df.loc[ticker]
            break   # Success

        except Exception as e:
            wait = RETRY_BACKOFF_BASE ** attempt
            print(f'{ticker} attempt {attempt} failed ({e}). Retrying in {wait}s...')
            time.sleep(wait)

    if df is None or df.empty:
        print(f'FAILED: {ticker}')
        failed.append(ticker)
        continue

    # Normalise column names to lower-case
    df.columns = [c.lower() for c in df.columns]
    df = df[['open', 'high', 'low', 'close', 'volume']]
    df.index = pd.to_datetime(df.index, utc=True)

    # Forward-fill small gaps (≤ FORWARD_FILL_LIMIT consecutive missing bars)
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='1min', tz='UTC')
    df = df.reindex(full_idx)
    df = df.ffill(limit=FORWARD_FILL_LIMIT)
    df = df.dropna()   # Drop any gaps larger than the fill limit

    # Save as compressed parquet
    df.to_parquet(out_path, compression='snappy')

    summary.append({
        'ticker'    : ticker,
        'rows'      : len(df),
        'start'     : str(df.index.min()),
        'end'       : str(df.index.max()),
    })

    # Polite sleep to stay within rate limits (200 req/min free tier)
    time.sleep(0.35)

print('\n=== SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if failed:
    print(f'\n⚠️  Failed tickers ({len(failed)}): {failed}')
else:
    print('\n✅ All tickers fetched successfully.')